In [1]:
import sys
import numpy as np
import tensorflow as tf
import tensorflow.keras as K
from pathlib import Path
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint
from sklearn.metrics import mean_squared_error, mean_absolute_error
from scipy.stats import pearsonr

2026-02-05 08:33:28.638712: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-02-05 08:33:28.648060: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:485] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
2026-02-05 08:33:28.659002: E external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:8473] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
2026-02-05 08:33:28.662104: E external/local_xla/xla/stream_executor/cuda/cuda_blas.cc:1471] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2026-02-05 08:33:28.670767: I tensorflow/core/platform/cpu_feature_guar

In [2]:
ROOT_DIR = Path.cwd().parents[1]
sys.path.append(str(ROOT_DIR / "src" / "data_preprocessing"))
from normalize_fn import load

In [3]:
hidden_layers=[1024,1024]
epochs=1000
act_func=tf.nn.relu
input_dropout=0.2
hidden_dropout=0.5
learning_rate=0.0001
norm='norm'

In [4]:
X_tr, X_val, _, _, y_tr, y_val, _, _ = load(norm=norm)

print("Training data shape:", X_tr.shape)
print("Validation data shape:", X_val.shape)
print("Training targets shape:", y_tr.shape)
print("Validation targets shape:", y_val.shape)

print("NaN in X_tr:", np.isnan(X_tr).any())
print("Inf in X_tr:", np.isinf(X_tr).any())
print("NaN in y_tr:", np.isnan(y_tr).any())
print("Inf in y_tr:", np.isinf(y_tr).any())

Training data shape: (13884, 7063)
Validation data shape: (4614, 7063)
Training targets shape: (13884, 1)
Validation targets shape: (4614, 1)
NaN in X_tr: False
Inf in X_tr: False
NaN in y_tr: False
Inf in y_tr: False


In [5]:
model=Sequential()
for i,units in enumerate(hidden_layers):
    if i==0:
        model.add(Dense(
            units,
            input_shape=(X_tr.shape[1],),
            activation=act_func,
            kernel_initializer='he_normal'))
        if input_dropout>0:
            model.add(Dropout(float(input_dropout)))
    else:
        model.add(Dense(
            units, 
            activation=act_func, 
            kernel_initializer='he_normal'))
        if hidden_dropout>0:
            model.add(Dropout(float(hidden_dropout)))
model.add(Dense(
    1,activation='linear',
    kernel_initializer='he_normal'))


I0000 00:00:1770280415.687458  132428 cuda_executor.cc:1001] could not open file to read NUMA node: /sys/bus/pci/devices/0000:02:00.0/numa_node
Your kernel may have been built without NUMA support.
I0000 00:00:1770280415.736123  132428 cuda_executor.cc:1001] could not open file to read NUMA node: /sys/bus/pci/devices/0000:02:00.0/numa_node
Your kernel may have been built without NUMA support.
I0000 00:00:1770280415.736159  132428 cuda_executor.cc:1001] could not open file to read NUMA node: /sys/bus/pci/devices/0000:02:00.0/numa_node
Your kernel may have been built without NUMA support.
I0000 00:00:1770280415.738287  132428 cuda_executor.cc:1001] could not open file to read NUMA node: /sys/bus/pci/devices/0000:02:00.0/numa_node
Your kernel may have been built without NUMA support.
I0000 00:00:1770280415.738310  132428 cuda_executor.cc:1001] could not open file to read NUMA node: /sys/bus/pci/devices/0000:02:00.0/numa_node
Your kernel may have been built without NUMA support.
I0000 00:0

In [6]:
optimizer=tf.keras.optimizers.SGD(learning_rate=learning_rate,momentum=0.5)
model.compile(loss='mean_squared_error',optimizer=optimizer)
model.summary()

Model: "sequential"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 dense (Dense)               (None, 1024)              7233536   
                                                                 
 dropout (Dropout)           (None, 1024)              0         
                                                                 
 dense_1 (Dense)             (None, 1024)              1049600   
                                                                 
 dropout_1 (Dropout)         (None, 1024)              0         
                                                                 
 dense_2 (Dense)             (None, 1)                 1025      
                                                                 
Total params: 8284161 (31.60 MB)
Trainable params: 8284161 (31.60 MB)
Non-trainable params: 0 (0.00 Byte)
_________________________________________________________________


In [7]:
# Callbacks
checkpoint_dir = Path("checkpoints")
checkpoint_dir.mkdir(exist_ok=True)
checkpoint_path = Path("checkpoints/final_model.h5")

callbacks = [
    EarlyStopping(
        monitor='val_loss', 
        patience=150, 
        restore_best_weights=True, 
        verbose=1
    ),
    ModelCheckpoint(
        filepath=str(checkpoint_path),
        monitor='val_loss',
        save_best_only=True,
        save_weights_only=False, # Save full model
        verbose=1
    )
]

In [8]:
history = model.fit(
    X_tr,y_tr,
    validation_data=(X_val,y_val),
    epochs=epochs,
    batch_size=64,
    callbacks=callbacks,
    shuffle=True,
    verbose=1
)

Epoch 1/1000


'+ptx85' is not a recognized feature for this target (ignoring feature)
'+ptx85' is not a recognized feature for this target (ignoring feature)
'+ptx85' is not a recognized feature for this target (ignoring feature)
'+ptx85' is not a recognized feature for this target (ignoring feature)
'+ptx85' is not a recognized feature for this target (ignoring feature)
'+ptx85' is not a recognized feature for this target (ignoring feature)
'+ptx85' is not a recognized feature for this target (ignoring feature)
'+ptx85' is not a recognized feature for this target (ignoring feature)
'+ptx85' is not a recognized feature for this target (ignoring feature)
'+ptx85' is not a recognized feature for this target (ignoring feature)
'+ptx85' is not a recognized feature for this target (ignoring feature)
'+ptx85' is not a recognized feature for this target (ignoring feature)
'+ptx85' is not a recognized feature for this target (ignoring feature)
'+ptx85' is not a recognized feature for this target (ignoring f

 55/217 [======>.......................] - ETA: 0s - loss: 491.5892 

'+ptx85' is not a recognized feature for this target (ignoring feature)
'+ptx85' is not a recognized feature for this target (ignoring feature)
'+ptx85' is not a recognized feature for this target (ignoring feature)


210/217 [============================>.] - ETA: 0s - loss: 373.1576
Epoch 1: val_loss improved from inf to 159.14374, saving model to checkpoints/final_model.h5


/usr/local/lib/python3.12/dist-packages/tf_keras/src/engine/training.py:3098: UserWarning: You are saving your model as an HDF5 file via `model.save()`. This file format is considered legacy. We recommend using instead the native TF-Keras format, e.g. `model.save('my_model.keras')`.
  saving_api.save_model(


217/217 [==============================] - 3s 6ms/step - loss: 366.7985 - val_loss: 159.1437
Epoch 2/1000
208/217 [===========================>..] - ETA: 0s - loss: 115.2005
Epoch 2: val_loss improved from 159.14374 to 43.67007, saving model to checkpoints/final_model.h5
217/217 [==============================] - 1s 5ms/step - loss: 113.3455 - val_loss: 43.6701
Epoch 3/1000
211/217 [============================>.] - ETA: 0s - loss: 53.4929
Epoch 3: val_loss improved from 43.67007 to 35.32812, saving model to checkpoints/final_model.h5
217/217 [==============================] - 1s 5ms/step - loss: 53.2748 - val_loss: 35.3281
Epoch 4/1000
207/217 [===========================>..] - ETA: 0s - loss: 36.2355
Epoch 4: val_loss improved from 35.32812 to 20.53753, saving model to checkpoints/final_model.h5
217/217 [==============================] - 1s 5ms/step - loss: 36.0265 - val_loss: 20.5375
Epoch 5/1000
208/217 [===========================>..] - ETA: 0s - loss: 27.3042
Epoch 5: val_loss im

In [9]:
print("Final training loss:",history.history['loss'][-1])
print("Final validation loss:",history.history['val_loss'][-1])

print(f"\nBest Training Loss: {min(history.history['loss'])}")
print(f"\nBest Validation Loss: {min(history.history['val_loss'])}")

Final training loss: 2.176189422607422
Final validation loss: 0.7641897797584534

Best Training Loss: 1.9618889093399048

Best Validation Loss: 0.5046216249465942


In [11]:
if checkpoint_path.exists():
    print("Loading best model from checkpoint...")
    model = tf.keras.models.load_model(str(checkpoint_path))
else:
    print("Checkpoint not found. Using current model.")

# Predict first, then flatten
y_val_pred = model.predict(X_val)
if y_val.ndim > 1:
    y_val_flat = y_val.flatten()
else:
    y_val_flat = y_val
if y_val_pred.ndim > 1:
    y_val_pred_flat = y_val_pred.flatten()
else:
    y_val_pred_flat = y_val_pred

mae = mean_absolute_error(y_val_flat, y_val_pred_flat)
mse = mean_squared_error(y_val_flat, y_val_pred_flat)
rmse = np.sqrt(mse)
pearson_corr, _ = pearsonr(y_val_flat, y_val_pred_flat)
print("Validation Metrics:")
print(f"Mean absolute error               :{mae:.4f}")
print(f"Mean squared error                :{mse:.4f}")
print(f"Root mean squared error           :{rmse:.4f}")
print(f"Pearson's correlation coefficient :{pearson_corr:.4f}")

Loading best model from checkpoint...
145/145 [==============================] - 0s 1ms/step
Validation Metrics:
Mean absolute error               :0.4545
Mean squared error                :0.5046
Root mean squared error           :0.7104
Pearson's correlation coefficient :0.9995
